In [23]:
import pandas as pd
import re
import unicodedata
import os
from pathlib import Path

In [24]:
# ===== PATH SETUP =====
BASE_DIR = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline")

INPUT_PATH = BASE_DIR / "data_inputs" / "raw_cv" / "cv_raw.xlsx"
OUTPUT_DIR = BASE_DIR / "data_outputs" / "step01_clean_text"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "01_cv_cleaned.xlsx"

raw_cv_dir = BASE_DIR / "data_inputs" / "raw_cv"
print("Folder exists:", raw_cv_dir.exists())
print("Files in folder:", os.listdir(raw_cv_dir) if raw_cv_dir.exists() else "Folder not found")

print(INPUT_PATH)
print("Exists:", INPUT_PATH.exists())
print("Suffix:", INPUT_PATH.suffix)
print("Size:", INPUT_PATH.stat().st_size if INPUT_PATH.exists() else "File not found")


Folder exists: True
Files in folder: ['.DS_Store', 'cv_raw.xlsx']
/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_inputs/raw_cv/cv_raw.xlsx
Exists: True
Suffix: .xlsx
Size: 6052


In [25]:
df = pd.read_excel(INPUT_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (20, 2)
Columns: ['candidate_id', 'cv_text']


,candidate_id,cv_text
0,C001,"Frontend developer experienced in ReactJS, Vue..."
1,C002,"Backend engineer experienced in Java, Spring B..."
2,C003,"Data analyst experienced in Python, pandas, Nu..."
3,C004,"DevOps engineer experienced in Linux, Docker, ..."
4,C005,"QA engineer experienced in manual testing, tes..."


In [26]:
df = pd.read_excel(INPUT_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (20, 2)
Columns: ['candidate_id', 'cv_text']


,candidate_id,cv_text
0,C001,"Frontend developer experienced in ReactJS, Vue..."
1,C002,"Backend engineer experienced in Java, Spring B..."
2,C003,"Data analyst experienced in Python, pandas, Nu..."
3,C004,"DevOps engineer experienced in Linux, Docker, ..."
4,C005,"QA engineer experienced in manual testing, tes..."


In [27]:
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    
    text = str(text)

    # chuẩn hóa unicode
    text = unicodedata.normalize("NFKC", text)

    # lowercase
    text = text.lower()

    # đổi xuống dòng / tab thành khoảng trắng
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")

    # bỏ ký tự đặc biệt rác, nhưng giữ lại một số ký hiệu hữu ích
    text = re.sub(r"[^a-zA-Z0-9À-ỹ\s\+\#\.\-/]", " ", text)

    # chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [28]:
df["cv_text_raw"] = df["cv_text"].astype(str)
df["cv_text_clean"] = df["cv_text_raw"].apply(clean_text)

display(df[["candidate_id", "cv_text_raw", "cv_text_clean"]].head())

,candidate_id,cv_text_raw,cv_text_clean
0,C001,"Frontend developer experienced in ReactJS, Vue...",frontend developer experienced in reactjs vuej...
1,C002,"Backend engineer experienced in Java, Spring B...",backend engineer experienced in java spring bo...
2,C003,"Data analyst experienced in Python, pandas, Nu...",data analyst experienced in python pandas nump...
3,C004,"DevOps engineer experienced in Linux, Docker, ...",devops engineer experienced in linux docker je...
4,C005,"QA engineer experienced in manual testing, tes...",qa engineer experienced in manual testing test...


In [29]:
df["raw_length"] = df["cv_text_raw"].apply(lambda x: len(str(x)))
df["clean_length"] = df["cv_text_clean"].apply(len)
df["is_empty_clean"] = df["cv_text_clean"].eq("")

display(df[["candidate_id", "raw_length", "clean_length", "is_empty_clean"]].head())
print("Số CV rỗng sau clean:", df["is_empty_clean"].sum())

,candidate_id,raw_length,clean_length,is_empty_clean
0,C001,249,240,False
1,C002,188,181,False
2,C003,163,154,False
3,C004,173,165,False
4,C005,172,166,False


Số CV rỗng sau clean: 0


In [30]:
## loại bỏ CV rỗng sau khi clean

df_cleaned = df.copy()
df_cleaned = df_cleaned[df_cleaned["cv_text_clean"].str.strip() != ""].reset_index(drop=True)

print("Số dòng sau khi bỏ CV rỗng:", len(df_cleaned))
display(df_cleaned.head())

Số dòng sau khi bỏ CV rỗng: 20


,candidate_id,cv_text,cv_text_raw,cv_text_clean,raw_length,clean_length,is_empty_clean
0,C001,"Frontend developer experienced in ReactJS, Vue...","Frontend developer experienced in ReactJS, Vue...",frontend developer experienced in reactjs vuej...,249,240,False
1,C002,"Backend engineer experienced in Java, Spring B...","Backend engineer experienced in Java, Spring B...",backend engineer experienced in java spring bo...,188,181,False
2,C003,"Data analyst experienced in Python, pandas, Nu...","Data analyst experienced in Python, pandas, Nu...",data analyst experienced in python pandas nump...,163,154,False
3,C004,"DevOps engineer experienced in Linux, Docker, ...","DevOps engineer experienced in Linux, Docker, ...",devops engineer experienced in linux docker je...,173,165,False
4,C005,"QA engineer experienced in manual testing, tes...","QA engineer experienced in manual testing, tes...",qa engineer experienced in manual testing test...,172,166,False


In [31]:
## lưu output

output_cols = [
    "candidate_id",
    "cv_text_raw",
    "cv_text_clean",
    "raw_length",
    "clean_length",
    "is_empty_clean"
]

df_cleaned[output_cols].to_excel(OUTPUT_PATH, index=False)
print("Đã lưu file:", OUTPUT_PATH)

Đã lưu file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step01_clean_text/01_cv_cleaned.xlsx


In [32]:
result = pd.read_excel(OUTPUT_PATH)
print("Output shape:", result.shape)
display(result.head())

Output shape: (20, 6)


,candidate_id,cv_text_raw,cv_text_clean,raw_length,clean_length,is_empty_clean
0,C001,"Frontend developer experienced in ReactJS, Vue...",frontend developer experienced in reactjs vuej...,249,240,False
1,C002,"Backend engineer experienced in Java, Spring B...",backend engineer experienced in java spring bo...,188,181,False
2,C003,"Data analyst experienced in Python, pandas, Nu...",data analyst experienced in python pandas nump...,163,154,False
3,C004,"DevOps engineer experienced in Linux, Docker, ...",devops engineer experienced in linux docker je...,173,165,False
4,C005,"QA engineer experienced in manual testing, tes...",qa engineer experienced in manual testing test...,172,166,False
